# Tuned Baseline: Fair Random Forest with Nested Threshold Tuning

LRNG: National (India) | CC: Human Values | SDG: 3 | Data: Time series | Thrust: Intelligent Edge Technologies | Screening aid only

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
import data_loading.v2_loader as v2

np.random.seed(42)

Loading PADS 100 Hz handcrafted features, cohort metadata, and 3-split fold assignments

In [2]:
pads_features = np.load(os.path.join("..", "results", "pads_features_100hz.npy"))
_, pads_meta = v2.load_v1_pads_data(results_dir=os.path.join("..", "results"))
splits = v2.load_fold_splits(results_dir=os.path.join("..", "results"))

y_pads = pads_meta["cohort_label"].map({"Normal": 0, "Parkinsons-like": 1, "ET-like": 2}).values
groups = pads_meta["subject_id"].values
unique_subs = pads_meta["subject_id"].unique()
sub_true = np.array([y_pads[groups == s][0] for s in unique_subs])
target_names = ["Normal", "Parkinsons-like", "ET-like"]

print("Loaded PADS features shape:", pads_features.shape)
print("Total subjects:", len(unique_subs))
print("Subject counts per class:", pd.Series(sub_true).value_counts().to_dict())

Loaded PADS features shape: (25500, 30)
Total subjects: 375
Subject counts per class: {1: 268, 0: 79, 2: 28}


Running nested decision threshold tuning on inner training folds across 3 split seeds (15 folds total)

In [3]:
split_seeds = [42, 43, 44]
seed_sub_raw_probs = []
seed_sub_scaled_probs = []
seed_win_scaled_probs = []
seed_sub_f1 = []
seed_sub_bacc = []

w0_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
w2_grid = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]

for s in split_seeds:
    oof_raw_probs = np.zeros((len(y_pads), 3), dtype=np.float32)
    oof_scaled_probs = np.zeros((len(y_pads), 3), dtype=np.float32)
    
    for fold in range(5):
        tr_idx = splits[f"split_{s}_tr_{fold}"]
        te_idx = splits[f"split_{s}_te_{fold}"]
        
        inner_sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=s)
        inner_oof_probs = np.zeros((len(tr_idx), 3), dtype=np.float32)
        
        for in_tr_local, in_va_local in inner_sgkf.split(pads_features[tr_idx], y_pads[tr_idx], groups[tr_idx]):
            in_tr = tr_idx[in_tr_local]
            in_va = tr_idx[in_va_local]
            rf_in = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, random_state=s, n_jobs=-1, class_weight="balanced")
            rf_in.fit(pads_features[in_tr], y_pads[in_tr])
            inner_oof_probs[in_va_local] = rf_in.predict_proba(pads_features[in_va])
            
        inner_subs = np.unique(groups[tr_idx])
        inner_sub_true = np.array([y_pads[groups == sub][0] for sub in inner_subs])
        inner_sub_p = np.array([np.mean(inner_oof_probs[groups[tr_idx] == sub], axis=0) for sub in inner_subs])
        
        best_inner_f1 = -1.0
        best_w = (1.0, 1.0, 1.0)
        for w0 in w0_grid:
            for w2 in w2_grid:
                scaled_p = inner_sub_p * np.array([w0, 1.0, w2])
                pred_c = np.argmax(scaled_p, axis=1)
                score = f1_score(inner_sub_true, pred_c, average="macro")
                if score > best_inner_f1:
                    best_inner_f1 = score
                    best_w = (w0, 1.0, w2)
                    
        rf_fold = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, random_state=s, n_jobs=-1, class_weight="balanced")
        rf_fold.fit(pads_features[tr_idx], y_pads[tr_idx])
        te_p = rf_fold.predict_proba(pads_features[te_idx])
        oof_raw_probs[te_idx] = te_p
        oof_scaled_probs[te_idx] = te_p * np.array(best_w)
        
    sub_p_raw = np.array([np.mean(oof_raw_probs[groups == sub], axis=0) for sub in unique_subs])
    sub_p_scaled = np.array([np.mean(oof_scaled_probs[groups == sub], axis=0) for sub in unique_subs])
    sub_pred = np.argmax(sub_p_scaled, axis=1)
    
    s_f1 = float(f1_score(sub_true, sub_pred, average="macro"))
    s_bacc = float(balanced_accuracy_score(sub_true, sub_pred))
    seed_sub_f1.append(s_f1)
    seed_sub_bacc.append(s_bacc)
    seed_sub_raw_probs.append(sub_p_raw)
    seed_sub_scaled_probs.append(sub_p_scaled)
    seed_win_scaled_probs.append(oof_scaled_probs)
    print(f"Split seed {s}: Subject Macro-F1 = {s_f1:.4f}, Subject Balanced Acc = {s_bacc:.4f}")

Split seed 42: Subject Macro-F1 = 0.5863, Subject Balanced Acc = 0.6702


Split seed 43: Subject Macro-F1 = 0.5619, Subject Balanced Acc = 0.6660


Split seed 44: Subject Macro-F1 = 0.5780, Subject Balanced Acc = 0.6601


Computing seed-averaged pooled metrics, bootstrap confidence intervals, and One-vs-Rest AUROCs

In [4]:
avg_sub_scaled_probs = np.mean(seed_sub_scaled_probs, axis=0)
avg_sub_raw_probs = np.mean(seed_sub_raw_probs, axis=0)
final_sub_preds = np.argmax(avg_sub_scaled_probs, axis=1)

avg_win_scaled_probs = np.mean(seed_win_scaled_probs, axis=0)
final_win_preds = np.argmax(avg_win_scaled_probs, axis=1)

mean_sub_f1 = float(f1_score(sub_true, final_sub_preds, average="macro"))
mean_sub_bacc = float(balanced_accuracy_score(sub_true, final_sub_preds))
mean_win_f1 = float(f1_score(y_pads, final_win_preds, average="macro"))
mean_win_bacc = float(balanced_accuracy_score(y_pads, final_win_preds))

auc_normal = float(roc_auc_score(sub_true == 0, avg_sub_raw_probs[:, 0]))
auc_pd = float(roc_auc_score(sub_true == 1, avg_sub_raw_probs[:, 1]))
auc_et = float(roc_auc_score(sub_true == 2, avg_sub_raw_probs[:, 2]))
auc_macro = float((auc_normal + auc_pd + auc_et) / 3.0)

f1_ci = v2.compute_bootstrap_ci(sub_true, avg_sub_scaled_probs, lambda yt, p: f1_score(yt, np.argmax(p, axis=1), average="macro"))
bacc_ci = v2.compute_bootstrap_ci(sub_true, avg_sub_scaled_probs, lambda yt, p: balanced_accuracy_score(yt, np.argmax(p, axis=1)))

print("=== Tuned Random Forest Pooled Results across 3 Split Seeds ===")
print(f"Subject Macro-F1 across 3 split seeds: {np.mean(seed_sub_f1):.4f} +- {np.std(seed_sub_f1):.4f}")
print(f"Pooled Subject Macro-F1: {mean_sub_f1:.4f} [95% Bootstrap CI: {f1_ci[0]:.4f} - {f1_ci[1]:.4f}]")
print(f"Pooled Subject Balanced Acc: {mean_sub_bacc:.4f} [95% Bootstrap CI: {bacc_ci[0]:.4f} - {bacc_ci[1]:.4f}]")
print(f"Subject One-vs-Rest AUROC: Normal={auc_normal:.4f}, PD={auc_pd:.4f}, ET={auc_et:.4f}, Macro={auc_macro:.4f}")
print(f"Window-Level Macro-F1: {mean_win_f1:.4f} | Balanced Acc: {mean_win_bacc:.4f}")

cm = confusion_matrix(sub_true, final_sub_preds)
print("\nSubject Confusion Matrix:")
print(pd.DataFrame(cm, index=[f"True_{c}" for c in target_names], columns=[f"Pred_{c}" for c in target_names]))
for i, c in enumerate(target_names):
    rec = float(cm[i, i] / np.sum(cm[i]))
    print(f"Recall for {c}: {rec:.4f}")
    
v2_rf_record = [{
    "Model": "Tuned Random Forest (Nested Thresholds)",
    "Sampling_Rate": "100 Hz",
    "Channels": 6,
    "Subject_Macro_F1": mean_sub_f1,
    "Subject_F1_CI_Low": f1_ci[0],
    "Subject_F1_CI_High": f1_ci[1],
    "Subject_Balanced_Acc": mean_sub_bacc,
    "Subject_BAcc_CI_Low": bacc_ci[0],
    "Subject_BAcc_CI_High": bacc_ci[1],
    "AUROC_Normal": auc_normal,
    "AUROC_PD": auc_pd,
    "AUROC_ET": auc_et,
    "AUROC_Macro": auc_macro,
    "Window_Macro_F1": mean_win_f1,
    "Window_Balanced_Acc": mean_win_bacc
}]
os.makedirs(os.path.join("..", "results", "metrics"), exist_ok=True)
pd.DataFrame(v2_rf_record).to_csv(os.path.join("..", "results", "metrics", "v2_track1_tuned_rf.csv"), index=False)
print("\nSaved results to results/metrics/v2_track1_tuned_rf.csv successfully")

=== Tuned Random Forest Pooled Results across 3 Split Seeds ===
Subject Macro-F1 across 3 split seeds: 0.5754 +- 0.0101
Pooled Subject Macro-F1: 0.5935 [95% Bootstrap CI: 0.5283 - 0.6511]
Pooled Subject Balanced Acc: 0.6816 [95% Bootstrap CI: 0.6070 - 0.7495]
Subject One-vs-Rest AUROC: Normal=0.8464, PD=0.7780, ET=0.8882, Macro=0.8375
Window-Level Macro-F1: 0.4662 | Balanced Acc: 0.5515

Subject Confusion Matrix:
                      Pred_Normal  Pred_Parkinsons-like  Pred_ET-like
True_Normal                    62                    16             1
True_Parkinsons-like           55                   175            38
True_ET-like                    1                    10            17
Recall for Normal: 0.7848
Recall for Parkinsons-like: 0.6530
Recall for ET-like: 0.6071

Saved results to results/metrics/v2_track1_tuned_rf.csv successfully


Comparing untuned vs tuned Random Forest baselines

In [5]:
v1_rf = pd.read_csv(os.path.join("..", "results", "metrics", "track1_baselines.csv"))
v2_rf = pd.read_csv(os.path.join("..", "results", "metrics", "v2_track1_tuned_rf.csv"))
comp_df = pd.concat([v1_rf, v2_rf], ignore_index=True)
display_cols = ["Model", "Subject_Macro_F1", "Subject_F1_CI_Low", "Subject_F1_CI_High", "Subject_Balanced_Acc", "AUROC_Macro"]
print(comp_df[display_cols].to_string(index=False))

                                  Model  Subject_Macro_F1  Subject_F1_CI_Low  Subject_F1_CI_High  Subject_Balanced_Acc  AUROC_Macro
               Random Forest (Baseline)          0.389012           0.324987            0.453426              0.392365     0.844024
Tuned Random Forest (Nested Thresholds)          0.593491           0.528285            0.651126              0.681646     0.837528
